In [ ]:
# ============================================
# Imports
# ============================================

import os
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Subscriber Churn Analysis

This project analyzes customer subscription data to identify churn patterns, calculate business KPIs, and build customer risk segments based on churn-related behavior.

## Business Problem

The goal of this project is to analyze subscriber data to identify churn trends, calculate key performance indicators, and create risk segments that highlight which customers are most likely to leave.

## Data Understanding

This dataset contains customer information from a telecommunications company, including subscription details, charges, service type, payment method, and churn status.

Key features used in the analysis include:
- tenure
- MonthlyCharges
- TotalCharges
- Contract
- PaymentMethod
- InternetService
- Churn

In [ ]:
# ============================================
# Load Data
# ============================================

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")
files = os.path.join(path, "WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = pd.read_csv(files)
df.head()

In [ ]:
# ============================================
# Inspect Dataset
# ============================================
rows, cols = df.shape
print(f"Dataset has {rows} rows and {cols} columns.")
print("Column names:", df.columns.tolist())
print("Data types:\n", df.dtypes)
print("Missing values:\n", df.isnull().sum())
print("Unique values in 'Churn':", df['Churn'].unique())
print("Value counts for 'Churn':\n", df['Churn'].value_counts())



In [ ]:
# ============================================
# Data Cleaning
# ============================================

df_clean = df.copy()

# convert columns if needed
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

    
# handle missing values
df_clean["TotalCharges"] = df_clean["TotalCharges"].fillna(df_clean["TotalCharges"].median())


# remove duplicates
df_clean.drop_duplicates(inplace=True)

# convert churn column to numeric
df_clean["Churn"] = df_clean["Churn"].map({"Yes": 1, "No": 0})

In [ ]:
# ============================================
# Feature Engineering
# ============================================

# create tenure buckets
df_clean["TenureBucket"] = pd.cut(
    df_clean["tenure"],
    bins=[0, 12, 24, 36, 48, 60, 72],
    labels=["0-12", "13-24", "25-36", "37-48", "49-60", "61-72"],
    include_lowest=True
)

# create charge buckets
df_clean["ChargeBucket"] = pd.cut(df_clean["MonthlyCharges"], bins=5, labels=["Low", "Medium", "High", "Very High", "Extremely High"])


In [ ]:
# ============================================
# KPI Calculation
# ============================================

# total customers
total_customers = len(df_clean)

# churn rate
churn_rate = df_clean["Churn"].mean()

# retention rate
retention_rate = 1 - churn_rate

# average revenue per user
avg_revenue_per_user = df_clean["MonthlyCharges"].mean()

# average tenure
avg_tenure = df_clean["tenure"].mean()

print(f"Total Customers:              {total_customers}")
print(f"Churn Rate:                   {churn_rate:.2%}")
print(f"Retention Rate:               {retention_rate:.2%}")
print(f"Avg Monthly Revenue per User: ${avg_revenue_per_user:.2f}")
print(f"Avg Tenure:                   {avg_tenure:.1f} months")

In [ ]:
# ============================================
# Exploratory Data Analysis
# ============================================

# churn distribution plot
sns.countplot(x="Churn", data=df_clean)
plt.title("Churn Distribution")
plt.show()

# churn by contract
sns.countplot(x="Contract", hue="Churn", data=df_clean)
plt.title("Churn by Contract Type")
plt.show()

# churn by tenure
sns.countplot(x="TenureBucket", hue="Churn", data=df_clean)
plt.title("Churn by Tenure Bucket")
plt.show()

# churn by payment method
sns.countplot(x="PaymentMethod", hue="Churn", data=df_clean)
plt.title("Churn by Payment Method")
plt.xticks(rotation=15)
plt.show()

# charges vs churn
sns.boxplot(x="Churn", y="MonthlyCharges", data=df_clean)
plt.title("Monthly Charges vs Churn")
plt.show()


In [ ]:
# ============================================
# Segment Analysis
# ============================================

# churn by contract
churn_by_contract = df_clean.groupby("Contract")["Churn"].mean()
print("Churn Rate by Contract:")
print(churn_by_contract)

# churn by tenure group
churn_by_tenure = df_clean.groupby("TenureBucket")["Churn"].mean()
print("\nChurn Rate by Tenure Bucket:")
print(churn_by_tenure)

# churn by payment method
churn_by_payment = df_clean.groupby("PaymentMethod")["Churn"].mean()
print("\nChurn Rate by Payment Method:")
print(churn_by_payment)

# churn by charge bucket
churn_by_charge = df_clean.groupby("ChargeBucket")["Churn"].mean()
print("\nChurn Rate by Charge Bucket:")
print(churn_by_charge)

In [ ]:
# ============================================
# Risk Segmentation
# ============================================

def assign_risk(row):
    # define rules for high / medium / low risk
    if row["Contract"] == "Month-to-month" and row["tenure"] < 12:
        return "High Risk"
    elif row["tenure"] < 24:
        return "Medium Risk"
    else:
        return "Low Risk"

df_clean["risk_segment"] = df_clean.apply(assign_risk, axis=1)

df_clean["risk_segment"].value_counts()



## Key Insights

- Customers on **month-to-month contracts have the highest churn rate** compared to longer-term contracts.
- Customers with **low tenure (new customers)** are significantly more likely to churn.
- Customers paying **higher monthly charges tend to churn more frequently** than those on lower-cost plans.
- Customers using **electronic check payment methods show higher churn rates** than automatic payment methods.

In [ ]:
# ============================================
# Export Processed Dataset
# ============================================

output_path = os.path.join("..", "data", "processed", "churn_cleaned.csv")
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df_clean.to_csv(output_path, index=False)
print("Saved to:", output_path)

In [ ]:
# Subscriber Churn Analysis

## 1. Business Problem
## 2. Load Dataset
## 3. Data Inspection
## 4. Data Cleaning
## 5. Feature Engineering
## 6. KPI Calculation
## 7. Exploratory Analysis
## 8. Segment Analysis
## 9. Risk Segmentation
## 10. Insights
## 11. Export Data